In [1]:
#문제 3번
import csv

class Heap:
	def __init__(self, *args):
		if len(args) != 0:
			self.__A = args[0] # 파라미터로 온 리스트
		else:
			self.__A = []
 
	# [알고리즘 8-2] 구현: 힙에 원소 삽입하기(재귀 알고리즘 버전)
	def insert(self, x):
		self.__A.append(x)
		self.__percolateUp(len(self.__A)-1)

	# 스며오르기
	def __percolateUp(self, i:int):
		parent = (i - 1) // 2
		if i > 0 and self.__A[i] > self.__A[parent]:
			self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
			self.__percolateUp(parent)

	# [알고리즘 8-2] 구현: 힙에서 원소 삭제하기
	def deleteMax(self):
		# heap is in self.__A[0...len(self.__A)-1]
		if (not self.isEmpty()):
			max = self.__A[0]
			self.__A[0] = self.__A.pop() # *.pop(): 리스트의 끝원소 삭제 후 원소 리턴
			self.__percolateDown(0)
			return max
		else:
			return None

	# 스며내리기
	def __percolateDown(self, i:int):
		# Percolate down w/ self.__A[i] as the root
		child = 2 * i + 1  # left child
		right = 2 * i + 2  # right child
		if (child <= len(self.__A)-1):
			if (right <= len(self.__A)-1 and self.__A[child] < self.__A[right]):
				child = right  # index of larger child
			if self.__A[i] < self.__A[child]:
				self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
				self.__percolateDown(child)

	def max(self):
		return self.__A[0]

	# 힙 만들기
	def buildHeap(self):
		for i in range((len(self.__A) - 2) // 2, -1, -1):
			self.__percolateDown(i)

	# 힙이 비었는지 확인하기
	def isEmpty(self) -> bool:
		return len(self.__A) == 0

	def clear(self):
		self.__A = []

	def size(self) -> int:
		return len(self.__A)

def print_youngest_10(file_path):
    h = Heap()
    
    with open(file_path, mode='r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        header = next(reader)  # 헤더 스킵
        
        print("--- 데이터 읽기 및 유효성 검사 ---")
        for row in reader:
            name = row[0].strip()
            year_str, month_str, day_str = row[1].strip(), row[2].strip(), row[3].strip()
            
            # 생년월일 값 중 하나라도 비어있는지 확인
            if not year_str or not month_str or not day_str:
                print(f"⚠️  {name}: 생년월일 정보가 비어있습니다.")
                continue
            
            try:
                # 숫자로 변환 (비교를 위해 YYYYMMDD 형식 생성)
                birth_val = int(year_str) * 10000 + int(month_str) * 100 + int(day_str)
                # Heap에 (날짜숫자, 이름) 튜플 형태로 삽입
                h.insert((birth_val, name))
            except ValueError:
                print(f"❌ {name}: 잘못된 형식의 데이터가 포함되어 있습니다.")

    print("\n--- 생년월일이 가장 느린(나이가 어린) 10명 ---")
    for i in range(1, 11):
        data = h.deleteMax() # Max Heap이므로 숫자가 큰(최신 날짜) 데이터부터 추출
        if data:
            birth_date, name = data
            # 출력 포맷팅
            date_str = f"{birth_date // 10000}-{ (birth_date % 10000) // 100:02d}-{birth_date % 100:02d}"
            print(f"{i}위: {name} ({date_str})")
        else:
            break

# 파일 실행
print_youngest_10('birthday.csv')

--- 데이터 읽기 및 유효성 검사 ---
⚠️  유지민: 생년월일 정보가 비어있습니다.

--- 생년월일이 가장 느린(나이가 어린) 10명 ---
1위: 이윤서 (2006-12-27)
2위: 정희원 (2006-12-21)
3위: 김효린 (2006-12-16)
4위: 이예은 (2006-12-09)
5위: 김주영 (2006-11-20)
6위: 전은빈 (2006-11-07)
7위: 이하연 (2006-09-22)
8위: 김우현 (2006-09-01)
9위: 최윤지 (2006-08-30)
10위: 유가현 (2006-08-22)


In [ ]:
#문제 4번

import csv

class BidirectNode:
    def __init__(self, x, prevNode:'BidirectNode', nextNode:'BidirectNode'):
        self.item = x
        self.prev = prevNode
        self.next = nextNode

class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0
 
    def insert(self, i:int, newItem) -> None:
        if (i >= 0 and i <= self.__numItems):
            prev = self.getNode(i - 1)
            newNode = BidirectNode(newItem, prev, prev.next)
            newNode.next.prev = newNode
            prev.next = newNode
            self.__numItems += 1
        else:
            print("index", i, ": out of bound in insert()") # 필요 시 에러 처리

    def append(self, newItem) -> None:
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def pop(self, *args):
        # 가변 파라미터. 인자가 없거나 -1이면 마지막 원소로 처리하기 위함. 파이썬 리스트 규칙 만족
        if self.isEmpty():
            return None
        # 인덱스 i 결정
        if len(args) != 0: # pop(k)과 같이 인자가 있으면 i = k 할당
            i = args[0]
        if len(args) == 0 or i == -1:# pop()에 인자가 없거나 pop(-1)이면 i에 맨 끝 인덱스 할당
            i = self.__numItems - 1
        # i번 원소 삭제
        if (i >= 0 and i <= self.__numItems - 1):
            curr = self.getNode(i)
            retItem = curr.item
            curr.prev.next = curr.next
            curr.next.prev = curr.prev
            self.__numItems -= 1
            return retItem
        else:
            return None
 
    def remove(self, x):
        curr = self.__findNode(x)
        if curr != None:
            curr.prev.next = curr.next
            curr.next.prev = curr.prev
            self.__numItems -= 1
            return x
        else:
            return None

    def get(self, *args):
        # 가변 파라미터. 인자가 없거나 -1이면 마지막 원소로 처리하기 위함. 파이썬 리스트 규칙 만족
        if self.isEmpty(): return None
        # 인덱스 i 결정
        if len(args) != 0:   # pop(k)과 같이 인자가 있으면 i = k 할당
            i = args[0]
        if len(args) == 0 or i == -1:# pop()에 인자가 없거나 pop(-1)이면 i에 맨 끝 인덱스 할당
            i = self.__numItems - 1
        # i번 원소 리턴
        if (i >= 0 and i <= self.__numItems - 1):
            return self.getNode(i).item
        else:
            return None
 
    def index(self, x) -> int:
        cnt = 0
        for element in self:
            if element == x:
                return cnt
            cnt += 1
        return -12345

    def isEmpty(self) -> bool:
        return self.__numItems == 0
 
    def size(self) -> int:
        return self.__numItems

    def clear(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0
 
    def count(self, x) -> int:
        cnt = 0
        for element in self:
            if element == x:
                    cnt += 1
        return cnt

    def extend(self, a): # a는 순회 가능한 모든 객체
        for element in a:
            self.append(element)

    def copy(self) -> 'CircularDoublyLinkedList':
        a = CircularDoublyLinkedList()
        for element in self:
            a.append(element)
        return a
 
    def reverse(self) -> None:
        prev = self.__head; curr = prev.next; next = curr.next
        self.__head.next = prev.prev; self.__head.prev = curr
        for i in range(self.__numItems):
            curr.next = prev; curr.prev = next
            prev = curr; curr = next; next = next.next
 
    def sort(self) -> None:
        a = []
        for element in self:
            a.append(element)
        a.sort()
        self.clear()
        for element in a:
            self.append(element)

    def __findNode(self, x) -> BidirectNode:
        curr = self.__head.next  # 0번 노드
        while curr != self.__head:
 
            if curr.item == x:
                return curr
            else:
                curr = curr.next
        return None

    def getNode(self, i:int) -> BidirectNode:
        curr = self.__head  # 더미 헤드, index: -1
        for index in range(i + 1):
            curr = curr.next
        return curr

    def printList(self) -> None:
        for element in self:
            print(element, end = ' ')
        print()

    def add(self, x) -> None:
        curr = self.__head.next  # 0번 노드부터 시작
        while curr != self.__head:
            if curr.item >= x:
                break
            curr = curr.next

        newNode = BidirectNode(x, curr.prev, curr)
        curr.prev.next = newNode
        curr.prev = newNode
        self.__numItems += 1

    def __iter__(self):  # generating iterator and return
        return CircularDoublyLinkedListIterator(self)
 
class CircularDoublyLinkedListIterator:
    def __init__(self, alist):
        self.__head = alist.getNode(-1)  		# 더미 헤드
        self.iterPosition = self.__head.next  	# 0번 노드
    def __next__(self):
        if self.iterPosition == self.__head: 	# 순환 끝
            raise StopIteration
        else: # 현재 원소를 리턴하면서 다음 원소로 이동
            item = self.iterPosition.item
            self.iterPosition = self.iterPosition.next
            return item
        
birthday_list = CircularDoublyLinkedList()

start_row = 33
end_row = 44

with open('birthday.csv', mode='r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    
    # enumerate를 사용하여 행 번호를 추적 (0부터 시작하므로 1을 더해 계산)
    current_idx = 0
    for i, row in enumerate(reader, start=1):
        if start_row <= i <= end_row:
            if any(row):  # 데이터가 있는 행인지 확인
                data = {
                    'name': row[0].strip(),
                    'year': row[1].strip(),
                    'month': row[2].strip(),
                    'day': row[3].strip()
                }
                birthday_list.insert(current_idx, data)
                current_idx += 1
        elif i > end_row:
            break  # 44행을 넘어가면 읽기 중단

# 3. 결과 출력
print(f"=== 같은 조 친구들의 명단 ===")
print(f"{'이름':<10} | {'생년':<6} | {'생월':<4} | {'생일':<4}")
print("-" * 40)

for person in birthday_list:
    print(f"{person['name']:<10} | {person['year']:<6} | {person['month']:<4} | {person['day']:<4}")

=== 같은 조 친구들의 명단 ===
이름         | 생년     | 생월   | 생일  
----------------------------------------
황다원        | 2004   | 10   | 15  
박다인        | 2005   | 1    | 19  
안예원        | 2004   | 7    | 9   
정서하        | 2006   | 7    | 20  
김우현        | 2006   | 9    | 1   
양시언        | 2006   | 4    | 18  
유가현        | 2006   | 8    | 22  
이하연        | 2006   | 9    | 22  
장예은        | 2006   | 3    | 17  
전은빈        | 2006   | 11   | 7   
정세희        | 2004   | 1    | 27  
이예은        | 2006   | 12   | 9   


문제 5번

1) 임의의 최대 힙에서 더 얕은 곳에 있는 원소가 더 깊은 곳에 있는 원소보다 더 작은 값을 가질 수 있다.
2) 최대 힙 A[0...n-1]에서 마지막 언소인 A[n-1]이 항상 가장 작은 값을 갖는 것은 아니다.
3) n/2개
4) 최악의 경우: Θ(n), 최선의 경우: Θ(1)
5) 간단하다. 단말노드만 삭제하면 되기 때문이다.
6) 문제에서 제시한 방법의 점근적 시간은 Θ(nlogn)이고, 본문에서 제시한 방법의 점근적 시간은 Θ(logn)이다. 따라서 본문의 방법이 더 효율적이다.
7) 늘어난 노드의 부모노드부터 heapify하면 된다. 삽입과 같은 과정을 거친다.